# Week 8 Bonus — Chart Your Data with Python

## What's this all about?

This is the **optional** companion to the Week 8 Quantified Self activity. You already tracked something about yourself in a Google Sheet and charted it there. Now let's do the same thing with *code*, so you can see how the charts you make in a spreadsheet are really just a friendly button on top of a few lines of Python.

You don't have to write much — mostly you'll paste your own numbers into one spot and run the cells.

> **Running this:** click a gray code cell and press **Shift+Enter**. Run top to bottom.

> **A note on accessibility:** charts are pictures, so this notebook prints a **text version of every chart** — bars drawn out of characters, with the real number on each row — alongside the matplotlib picture. The number summaries in Step 2 and the text bars in Steps 3–4 carry the whole lesson on their own, so you can follow along with a screen reader, a magnifier, or a Braille display without needing to see the images. (Even the "how a chart lies" demo in Step 4 works in text.)

## Step 1: Put your data in

Type your tracked numbers into the lists below. Put your day labels in `days` and your measurements in `values`, in the same order. (Replace my example step-count data with yours.) If you left a day blank in your sheet, type `None` in its spot, with no quotation marks. The code below skips those days when it does math and marks them as "no data" in the charts.

In [ ]:
days   = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
values = [6200,   6800,  6500,  15000, 6400,  7100,  6600]   # e.g. daily steps (Thu was a hike)

what_i_tracked = "Daily steps"   # <-- change this label to your thing

# A quick sanity check that the two lists line up:
print("Days recorded:", len(days))
print("Values recorded:", len(values))
if len(days) != len(values):
    print("⚠️  Your two lists are different lengths — fix that before charting!")
else:
    print("Looks good. On to the chart.")

## Step 2: Look at the numbers

Before any chart, just look. Python can hand you the basic shape of your data in a couple of lines:

In [ ]:
import statistics

logged = [v for v in values if v is not None]     # leave out any missed days
missed = len(values) - len(logged)

print(f"Tracking: {what_i_tracked}")
print("Days with data:", len(logged), f"(missed {missed})")
print("Highest day:   ", max(logged))
print("Lowest day:    ", min(logged))
print("Mean:          ", round(statistics.mean(logged), 1))
print("Median:        ", statistics.median(logged))

Lesson 4 of this unit showed two kinds of "typical value": the **mean** (add everything up and divide) and the **median** (the middle value once everything is lined up). In the example data, Thursday's 15,000-step hike pulls the mean up to 7,800 steps, but the median stays at 6,600, because the median only cares about what's in the middle. So which one better describes a typical day that week? Do your mean and median disagree? If they do, look for the outlier day, and check your Notes column for what happened.

## Step 3: Make a chart

A chart you can't read isn't much of a chart, so first let's draw your week as a **text bar chart** — bars made of characters, readable by eye, magnifier, or screen reader. We'll write a little helper and reuse it in Step 4 too.

In [ ]:
def text_bars(labels, values, low, high, width=40):
    """Draw a horizontal bar chart out of text characters.
    Each bar's length is scaled between `low` (empty) and `high` (full)."""
    span = high - low if high > low else 1
    for label, v in zip(labels, values):
        if v is None:
            print(f"{label:>4} | {'':<{width}} (no data)")
            continue
        frac = min(max((v - low) / span, 0), 1)
        bar = "#" * round(frac * width)
        print(f"{label:>4} | {bar:<{width}} (value {v})")

print(f"My week of: {what_i_tracked}\n")
text_bars(days, values, low=0, high=max(logged) + 1)

Each row reads as: the day, then a bar whose length matches that day's value, then the value itself in parentheses. Longer bar, bigger number.

Now the same data as a `matplotlib` chart, for anyone who'd also like the picture — a line chart first:

In [ ]:
import matplotlib.pyplot as plt

# matplotlib can't draw None, so missed days become "nan" (not a number),
# which it leaves as a gap.
plot_values = [float("nan") if v is None else v for v in values]

plt.figure(figsize=(8, 4))
plt.plot(days, plot_values, marker="o")
plt.title(f"My week of: {what_i_tracked}")
plt.xlabel("Day")
plt.ylabel(what_i_tracked)
plt.grid(True, alpha=0.3)
plt.show()

…and the bar version (`plt.bar` instead of `plt.plot`):

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(days, plot_values)
plt.title(f"My week of: {what_i_tracked}")
plt.xlabel("Day")
plt.ylabel(what_i_tracked)
plt.show()

## Step 4: How a chart can lie (the important part)

This is the **truncated y-axis** from Lesson 2, done to your own data. The trick is to start the bottom of the chart **not at zero**. We'll draw your data twice as text bars — once honestly (bars measured from 0) and once "sneakily" (bars measured from just below your smallest value). The `(value N)` at the end of each row is *identical* both times. Only the bar lengths change.

In [ ]:
print("HONEST — bars measured from 0:\n")
text_bars(days, values, low=0, high=max(logged) + 1)

print("\nSNEAKY — bars measured from just below the smallest value:\n")
margin = (max(logged) - min(logged)) * 0.05 or 0.5   # a little room below the smallest bar
text_bars(days, values, low=min(logged) - margin, high=max(logged) + margin)

Compare the two sets of bars. Same data — every `(value N)` is unchanged — but in the "sneaky" version the gaps between days look **enormous**, because we chopped the bottom off the scale. A small real difference gets stretched into a dramatic-looking one.

Here's that same deception drawn as matplotlib charts, side by side, if you'd like to see the picture:

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

left.bar(days, plot_values, color="steelblue")
left.set_title("Honest: y-axis starts at 0")
left.set_ylim(0, max(logged) + 1)

right.bar(days, plot_values, color="indianred")
right.set_title("Sneaky: y-axis starts near the bottom value")
right.set_ylim(min(logged) - margin, max(logged) + margin)

plt.show()

Same numbers. Same week. Nobody lied about a single value — they just chose where the axis starts. Lesson 2's Fox News tax-rate chart worked the same way, and the trick shows up in ads, news graphics, and companies' earnings slides too.

## Reflection (add to your Week 8 post if you do this bonus)

1. The two charts in Step 4 used identical data and told two different stories. Where have you seen a chart "in the wild" that you now suspect was using this trick?
2. Spreadsheets and Python both made these charts from your numbers with almost no effort. If making a misleading chart is this easy, whose job is it to catch it — the person who makes it, or the person who reads it?